# Paperspace ComfyUI Runtime

Notebook 側では設定とスクリプト実行だけを行います。モデル同期のロジックは `scripts/` に切り出しています。


In [ ]:
# Notebook 全体で使う設定値です。
# まずは MODEL_MODE だけ変えれば使える想定です。
from pathlib import Path
import os
import subprocess

REPO_ROOT = Path.cwd().resolve()
COMFYUI_DIR = "/storage/ComfyUI"
MODEL_ROOT = "/app/models"
HF_REPO_CONFIG = "/storage/ComfyUI/hf-repo.yaml"
MODEL_MODE = "image"  # image: loras のみ / video: loras 以外
FORCE_DOWNLOAD = False
COMFYUI_PORT = 6006
COMFYUI_ARGS = "--preview-method auto"
COMFYUI_PYTHON = ""  # 例: /storage/ComfyUI/.venv/bin/python
PAPERSPACE_FQDN = os.environ.get("PAPERSPACE_FQDN", "")
SCRIPTS_DIR = REPO_ROOT / "scripts"

print(f"REPO_ROOT={REPO_ROOT}")
print(f"COMFYUI_DIR={COMFYUI_DIR}")
print(f"MODEL_ROOT={MODEL_ROOT}")
print(f"HF_REPO_CONFIG={HF_REPO_CONFIG}")
print(f"MODEL_MODE={MODEL_MODE}")
print(f"COMFYUI_PORT={COMFYUI_PORT}")
print(f"SCRIPTS_DIR={SCRIPTS_DIR}")
if PAPERSPACE_FQDN:
    print(f"PUBLIC_COMFYUI_URL=https://tensorboard-{PAPERSPACE_FQDN}")


In [ ]:
# コンテナ内コマンドと ComfyUI 配置を先に検証します。
subprocess.run(
    [
        "python",
        str(SCRIPTS_DIR / "check_runtime.py"),
        "--comfyui-dir",
        COMFYUI_DIR,
        "--model-root",
        MODEL_ROOT,
    ],
    check=True,
)


In [ ]:
# Hugging Face 認証が見えているか確認します。
subprocess.run(["python", str(SCRIPTS_DIR / "check_hf_auth.py")], check=True)


In [ ]:
# repo 設定と MODEL_MODE に応じて必要なモデルだけ同期します。
cmd = [
    "python",
    str(SCRIPTS_DIR / "sync_hf_repo.py"),
    "--config",
    HF_REPO_CONFIG,
    "--model-root",
    MODEL_ROOT,
    "--mode",
    MODEL_MODE,
]
if FORCE_DOWNLOAD:
    cmd.append("--force")

subprocess.run(cmd, check=True)


In [ ]:
# /app/models を ComfyUI から見えるように extra_model_paths.yaml を再生成します。
subprocess.run(
    [
        "python",
        str(SCRIPTS_DIR / "write_extra_model_paths.py"),
        "--comfyui-dir",
        COMFYUI_DIR,
        "--model-root",
        MODEL_ROOT,
    ],
    check=True,
)


In [ ]:
# ComfyUI 本体を起動します。Paperspace では 6006 を tensorboard-* URL で開く前提です。
cmd = [
    "python",
    str(SCRIPTS_DIR / "run_comfyui.py"),
    "--comfyui-dir",
    COMFYUI_DIR,
    "--port",
    str(COMFYUI_PORT),
    "--args",
    COMFYUI_ARGS,
]
if COMFYUI_PYTHON:
    cmd.extend(["--python", COMFYUI_PYTHON])

subprocess.run(cmd, check=True)


In [ ]:
# 状況確認用のメモです。失敗時の確認にも使えます。
print("Notebook troubleshooting notes")
print(f"- Mode: {MODEL_MODE}")
print(f"- Repo config: {HF_REPO_CONFIG}")
print(f"- Models root: {MODEL_ROOT}")
print(f"- Paperspace FQDN: {PAPERSPACE_FQDN or '(not set)'}")
if PAPERSPACE_FQDN:
    print(f"- ComfyUI URL: https://tensorboard-{PAPERSPACE_FQDN}")
